In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
application = pd.read_csv("data/application_record.csv")
credit = pd.read_csv("data/credit_record.csv")

print("Application dataset shape:", application.shape)
print("Credit dataset shape:", credit.shape)

Application dataset shape: (438557, 18)
Credit dataset shape: (1048575, 3)


In [3]:
application.head()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1


In [4]:
credit.head()

,ID,MONTHS_BALANCE,STATUS
0,5001711,0,X
1,5001711,-1,0
2,5001711,-2,0
3,5001711,-3,0
4,5001712,0,C


In [5]:
credit["STATUS"].value_counts()

STATUS
C    442031
0    383120
X    209230
1     11090
5      1693
2       868
3       320
4       223
Name: count, dtype: int64

In [8]:
credit["Approved"] = credit["STATUS"].apply(
    lambda x: 0 if x in ["2", "3", "4", "5"] else 1
)

In [9]:
credit = credit.groupby("ID")["Approved"].min().reset_index()

credit.head()

,ID,Approved
0,5001711,1
1,5001712,1
2,5001713,1
3,5001714,1
4,5001715,1


In [10]:
df = application.merge(credit, on="ID")

print(df.shape)
df.head()

(36457, 19)


,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,Approved
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2,1
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2,1
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2,1
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1,1
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1,1


In [11]:
df.isnull().sum()

ID                         0
CODE_GENDER                0
FLAG_OWN_CAR               0
FLAG_OWN_REALTY            0
CNT_CHILDREN               0
AMT_INCOME_TOTAL           0
NAME_INCOME_TYPE           0
NAME_EDUCATION_TYPE        0
NAME_FAMILY_STATUS         0
NAME_HOUSING_TYPE          0
DAYS_BIRTH                 0
DAYS_EMPLOYED              0
FLAG_MOBIL                 0
FLAG_WORK_PHONE            0
FLAG_PHONE                 0
FLAG_EMAIL                 0
OCCUPATION_TYPE        11323
CNT_FAM_MEMBERS            0
Approved                   0
dtype: int64

In [12]:
df["OCCUPATION_TYPE"] = df["OCCUPATION_TYPE"].fillna("Unknown")

In [44]:
df.isnull().sum()

CODE_GENDER            0
FLAG_OWN_CAR           0
FLAG_OWN_REALTY        0
CNT_CHILDREN           0
AMT_INCOME_TOTAL       0
NAME_INCOME_TYPE       0
NAME_EDUCATION_TYPE    0
NAME_FAMILY_STATUS     0
NAME_HOUSING_TYPE      0
DAYS_BIRTH             0
DAYS_EMPLOYED          0
FLAG_MOBIL             0
FLAG_WORK_PHONE        0
FLAG_PHONE             0
FLAG_EMAIL             0
OCCUPATION_TYPE        0
CNT_FAM_MEMBERS        0
Approved               0
dtype: int64

In [46]:
df.drop("ID", axis=1, inplace=True, errors="ignore")

In [48]:
df.head()

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,Approved
0,1,1,1,0,427500.0,4,1,0,4,-12005,-4542,1,1,0,0,17,2,1
1,1,1,1,0,427500.0,4,1,0,4,-12005,-4542,1,1,0,0,17,2,1
2,1,1,1,0,112500.0,4,4,1,1,-21474,-1134,1,0,0,0,16,2,1
3,0,0,1,0,270000.0,0,4,3,1,-19110,-3051,1,0,1,1,14,1,1
4,0,0,1,0,270000.0,0,4,3,1,-19110,-3051,1,0,1,1,14,1,1


In [49]:
import pickle

label_encoders = {}

for col in df.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

with open("label_encoders.pkl", "wb") as f:
    pickle.dump(label_encoders, f)

print("Encoders saved successfully!")

Encoders saved successfully!


In [50]:
X = df.drop("Approved", axis=1)
y = df["Approved"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (36457, 17)
Target shape: (36457,)


In [51]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (29165, 17)
Testing data: (7292, 17)


In [52]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [53]:
y_pred = model.predict(X_test)

In [54]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print()

print(classification_report(y_test, y_pred))

Accuracy: 0.9673614920460779

              precision    recall  f1-score   support

           0       0.23      0.44      0.30       117
           1       0.99      0.98      0.98      7175

    accuracy                           0.97      7292
   macro avg       0.61      0.71      0.64      7292
weighted avg       0.98      0.97      0.97      7292



In [55]:
import pickle

pickle.dump(model, open("model.pkl", "wb"))

print("Model saved successfully!")

Model saved successfully!


In [56]:
categorical_columns = [
    "CODE_GENDER",
    "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY",
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE"
]

for col in categorical_columns:
    print(f"\n{col}")
    print(application[col].dropna().unique())


CODE_GENDER
[1 0]

FLAG_OWN_CAR
[1 0]

FLAG_OWN_REALTY
[1 0]

NAME_INCOME_TYPE
[4 0 1 2 3]

NAME_EDUCATION_TYPE
[1 4 2 3 0]

NAME_FAMILY_STATUS
[0 1 3 2 4]

NAME_HOUSING_TYPE
[4 1 2 5 0 3]

OCCUPATION_TYPE
[10  8  6  0 17  2 13 12 15  1  4 11 18  3  7  9 14  5 16]


In [57]:
import pickle

with open("label_encoders.pkl", "rb") as f:
    label_encoders = pickle.load(f)

print(label_encoders.keys())

dict_keys([])


In [58]:
from sklearn.preprocessing import LabelEncoder
import pickle

categorical_columns = [
    "CODE_GENDER",
    "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY",
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE"
]

label_encoders = {}

for col in categorical_columns:
    le = LabelEncoder()
    application[col] = le.fit_transform(application[col].astype(str))
    label_encoders[col] = le

with open("label_encoders.pkl", "wb") as f:
    pickle.dump(label_encoders, f)

print("Label encoders saved successfully!")

Label encoders saved successfully!


In [33]:
import pickle

with open("label_encoders.pkl","rb") as f:
    label_encoders = pickle.load(f)

print(label_encoders.keys())

dict_keys(['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE'])


In [59]:
credit["Approved"].value_counts()

Approved
1    45318
0      667
Name: count, dtype: int64

In [60]:
print(df.columns.tolist())

['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'FLAG_MOBIL', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS', 'Approved']


In [61]:
import pickle

with open("label_encoders.pkl", "rb") as f:
    encoders = pickle.load(f)

print(encoders.keys())

dict_keys(['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE'])


In [62]:
import pickle

with open("label_encoders.pkl", "rb") as f:
    encoders = pickle.load(f)

print(encoders["CODE_GENDER"].classes_)

['0' '1']


In [63]:
print(df["NAME_INCOME_TYPE"].unique())

[4 0 1 2 3]


In [64]:
print(application["NAME_INCOME_TYPE"].unique())

[4 0 1 2 3]


In [65]:
print(application["NAME_INCOME_TYPE"].unique())

[4 0 1 2 3]


In [66]:
print(application_record["NAME_INCOME_TYPE"].unique())

NameError: name 'application_record' is not defined

In [67]:
import pandas as pd

application = pd.read_csv("data/application_record.csv")
credit = pd.read_csv("data/credit_record.csv")

print(application.head())
print(credit.head())

        ID CODE_GENDER FLAG_OWN_CAR FLAG_OWN_REALTY  CNT_CHILDREN  \
0  5008804           M            Y               Y             0   
1  5008805           M            Y               Y             0   
2  5008806           M            Y               Y             0   
3  5008808           F            N               Y             0   
4  5008809           F            N               Y             0   

   AMT_INCOME_TOTAL      NAME_INCOME_TYPE            NAME_EDUCATION_TYPE  \
0          427500.0               Working               Higher education   
1          427500.0               Working               Higher education   
2          112500.0               Working  Secondary / secondary special   
3          270000.0  Commercial associate  Secondary / secondary special   
4          270000.0  Commercial associate  Secondary / secondary special   

     NAME_FAMILY_STATUS  NAME_HOUSING_TYPE  DAYS_BIRTH  DAYS_EMPLOYED  \
0        Civil marriage   Rented apartment      -12005 

In [68]:
# Create target variable from credit history

credit["Approved"] = credit["STATUS"].apply(
    lambda x: 0 if x in ["2", "3", "4", "5"] else 1
)

# Keep one record per customer
credit = credit.groupby("ID")["Approved"].min().reset_index()

# Merge datasets
df = application.merge(credit, on="ID", how="inner")

print(df.head())
print(df.shape)

        ID CODE_GENDER FLAG_OWN_CAR FLAG_OWN_REALTY  CNT_CHILDREN  \
0  5008804           M            Y               Y             0   
1  5008805           M            Y               Y             0   
2  5008806           M            Y               Y             0   
3  5008808           F            N               Y             0   
4  5008809           F            N               Y             0   

   AMT_INCOME_TOTAL      NAME_INCOME_TYPE            NAME_EDUCATION_TYPE  \
0          427500.0               Working               Higher education   
1          427500.0               Working               Higher education   
2          112500.0               Working  Secondary / secondary special   
3          270000.0  Commercial associate  Secondary / secondary special   
4          270000.0  Commercial associate  Secondary / secondary special   

     NAME_FAMILY_STATUS  NAME_HOUSING_TYPE  DAYS_BIRTH  DAYS_EMPLOYED  \
0        Civil marriage   Rented apartment      -12005 

In [69]:
df["OCCUPATION_TYPE"] = df["OCCUPATION_TYPE"].fillna("Unknown")

In [70]:
print(df["CODE_GENDER"].unique())
print(df["NAME_INCOME_TYPE"].unique())

<StringArray>
['M', 'F']
Length: 2, dtype: str
<StringArray>
['Working', 'Commercial associate', 'Pensioner', 'State servant', 'Student']
Length: 5, dtype: str


In [71]:
from sklearn.preprocessing import LabelEncoder
import pickle

categorical_cols = [
    "CODE_GENDER",
    "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY",
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE"
]

label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

with open("label_encoders.pkl", "wb") as f:
    pickle.dump(label_encoders, f)

print("Label encoders saved successfully!")

Label encoders saved successfully!


In [72]:
X = df.drop(["ID", "Approved"], axis=1)
y = df["Approved"]

In [73]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [74]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [75]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved successfully!")

Model saved successfully!


In [76]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.21      0.46      0.29       123
           1       0.99      0.97      0.98      7169

    accuracy                           0.96      7292
   macro avg       0.60      0.71      0.63      7292
weighted avg       0.98      0.96      0.97      7292

